# Hybrid Prophet+GRU vs Global GRU — Full Comparison (Read-Only)

**Module 1 methodology comparison.** Side-by-side evaluation of the two frozen forecasting architectures under an identical Day-1 protocol.

> **Scope:** Read-only view of `experiments/hybrid_vs_global_2026-07-17_095147/`.  
> No retraining. No baseline modification. Loads frozen evaluation tables, regenerates comparison plots, and states the locked research conclusion.  
> Source summary: `FINAL_COMPARISON_SUMMARY.md`  
> Docs: `docs/global-gru-baseline/`, Hybrid baseline `experiments/baseline_reference_2026-07-14/`

## Research questions

1. How does Global GRU compare to Hybrid on overall Day-1 accuracy?
2. Which methodology wins on which containers?
3. What are the deployment trade-offs?

## Δ convention

**Δ = Global − Hybrid**  
- Positive Δ → Global has **higher** error (Hybrid better)  
- Negative Δ → Global has **lower** error (Global better)

## Kernel

Use `tf_metal_env` / `FYP_Long_Term`. Run from `notebooks/` (paths use `Path("..")`).


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.global_comparison_plotting import (
    COLOR_GLOBAL,
    COLOR_HYBRID,
    COLOR_REFERENCE_LINE,
    select_comparison_sample_container_ids,
)

EXP_DIR = REPO_ROOT / "experiments" / "hybrid_vs_global_2026-07-17_095147"
PLOT_DIR = EXP_DIR / "plots"

required = [
    EXP_DIR / "comparison_table.csv",
    EXP_DIR / "per_container_delta.csv",
    EXP_DIR / "phase6_comparison.json",
    EXP_DIR / "FINAL_COMPARISON_SUMMARY.md",
    EXP_DIR / "experiment_config.json",
]
missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing frozen artefacts:\n" + "\n".join(map(str, missing)))

comparison = pd.read_csv(EXP_DIR / "comparison_table.csv")
deltas = pd.read_csv(EXP_DIR / "per_container_delta.csv")
with open(EXP_DIR / "phase6_comparison.json") as fh:
    phase6 = json.load(fh)
with open(EXP_DIR / "experiment_config.json") as fh:
    exp_cfg = json.load(fh)

print("Experiment :", EXP_DIR.relative_to(REPO_ROOT))
print("Containers :", phase6["evaluated_containers"])
print("Hybrid ref :", phase6["experiment_config"]["hybrid_frozen_reference"])
print("Global ref :", phase6["experiment_config"]["global_gru_frozen_reference"])
print("Δ convention:", phase6["delta_sign_convention"])


Experiment : experiments/hybrid_vs_global_2026-07-17_095147
Containers : 99
Hybrid ref : experiments/baseline_reference_2026-07-14
Global ref : experiments/global_gru_baseline_2026-07-17_121748
Δ convention: global_minus_hybrid


## 1. What is being compared?

Two **forecasting methodologies**, not two random hyperparameter settings.

| Dimension | Hybrid Prophet + GRU | Global GRU |
|-----------|----------------------|------------|
| Idea | Decompose: Prophet trend/seasonality + GRU residual correction | Direct end-to-end GRU on CPU sequences |
| Input signal | Residuals after Prophet | Scaled CPU (+ static context) |
| Input window | 96 steps (24 h) | 96 steps (24 h) |
| Horizon | Day-1 = 96 steps | Day-1 = 96 steps |
| Inference | Per-container Prophet fit + shared GRU | Single shared GRU forward pass |
| Frozen reference | `baseline_reference_2026-07-14` | `global_gru_baseline_2026-07-17_121748` |

**Intentional difference:** methodology only. Preprocessing, temporal split, metrics, and cohort are identical.


## 2. Shared evaluation protocol

In [2]:
protocol_rows = [
    ("Dataset", "Alibaba Cluster Trace"),
    ("Target", "cpu_util_percent (real CPU %)"),
    ("Resampling", "15-minute"),
    ("Input window", "96 timesteps (24 h)"),
    ("Day-1 horizon", "96 validation timesteps / container"),
    ("Cohort", "100 selected → 99 evaluable (c_14674 skipped)"),
    ("Temporal split", "Per-container 80/20 chronological holdout"),
    ("Primary metrics", "MAE, RMSE (MAPE secondary)"),
    ("MAPE ε", "0.01"),
    ("Δ convention", "Global − Hybrid"),
    ("Random seed", "42"),
]
display(Markdown("Table — Locked comparison protocol"))
display(pd.DataFrame(protocol_rows, columns=["Item", "Value"]))


Table — Locked comparison protocol

,Item,Value
0,Dataset,Alibaba Cluster Trace
1,Target,cpu_util_percent (real CPU %)
2,Resampling,15-minute
3,Input window,96 timesteps (24 h)
4,Day-1 horizon,96 validation timesteps / container
5,Cohort,100 selected → 99 evaluable (c_14674 skipped)
6,Temporal split,Per-container 80/20 chronological holdout
7,Primary metrics,"MAE, RMSE (MAPE secondary)"
8,MAPE ε,0.01
9,Δ convention,Global − Hybrid


## 3. Aggregate Day-1 accuracy

Primary ranking uses **mean Day-1 MAE / RMSE** across the 99-container cohort.


In [3]:
display(Markdown("Table — Aggregate methodology comparison"))
agg = comparison.copy()
agg["methodology"] = agg["methodology"].replace({
    "hybrid_prophet_gru": "Hybrid Prophet+GRU",
    "global_gru_v1": "Global GRU",
})
show_cols = [
    "methodology", "n_containers",
    "day1_mae_mean", "day1_mae_std",
    "day1_rmse_mean", "day1_rmse_std",
    "day1_mape_mean", "day1_mape_std",
]
display(agg[show_cols].round(4))

pr = phase6["primary_results"]
delta_tbl = pd.DataFrame([
    {
        "metric": "Day-1 MAE",
        "hybrid": pr["day1_mae"]["hybrid_mean"],
        "global": pr["day1_mae"]["global_mean"],
        "delta_G_minus_H": pr["day1_mae"]["mean_delta_global_minus_hybrid"],
        "winner": "Hybrid" if pr["day1_mae"]["mean_delta_global_minus_hybrid"] > 0 else "Global",
    },
    {
        "metric": "Day-1 RMSE",
        "hybrid": pr["day1_rmse"]["hybrid_mean"],
        "global": pr["day1_rmse"]["global_mean"],
        "delta_G_minus_H": pr["day1_rmse"]["mean_delta_global_minus_hybrid"],
        "winner": "Hybrid" if pr["day1_rmse"]["mean_delta_global_minus_hybrid"] > 0 else "Global",
    },
    {
        "metric": "Day-1 MAPE (secondary)",
        "hybrid": pr["day1_mape"]["hybrid_mean"],
        "global": pr["day1_mape"]["global_mean"],
        "delta_G_minus_H": pr["day1_mape"]["mean_delta_global_minus_hybrid"],
        "winner": "Hybrid" if pr["day1_mape"]["mean_delta_global_minus_hybrid"] > 0 else "Global",
    },
])
display(Markdown("Table — Mean deltas (Global − Hybrid)"))
display(delta_tbl.round(4))

print(
    f"Headline: Hybrid MAE {pr['day1_mae']['hybrid_mean']:.3f} "
    f"vs Global {pr['day1_mae']['global_mean']:.3f} "
    f"(Δ = {pr['day1_mae']['mean_delta_global_minus_hybrid']:+.3f})"
)


Table — Aggregate methodology comparison

,methodology,n_containers,day1_mae_mean,day1_mae_std,day1_rmse_mean,day1_rmse_std,day1_mape_mean,day1_mape_std
0,Hybrid Prophet+GRU,99,1.7459,2.4868,2.3878,3.2191,111.9403,615.2991
1,Global GRU,99,1.9242,2.3155,2.6105,2.9653,116.5026,642.9235


Table — Mean deltas (Global − Hybrid)

,metric,hybrid,global,delta_G_minus_H,winner
0,Day-1 MAE,1.7459,1.9242,0.1784,Hybrid
1,Day-1 RMSE,2.3878,2.6105,0.2227,Hybrid
2,Day-1 MAPE (secondary),111.9403,116.5026,4.5623,Hybrid


Headline: Hybrid MAE 1.746 vs Global 1.924 (Δ = +0.178)


## 4. Per-container win matrix

Count how often each methodology has lower Day-1 error on the same container.


In [4]:
ds = phase6["delta_summary"]

win_rows = []
for metric, label in [
    ("day1_mae", "Day-1 MAE"),
    ("day1_rmse", "Day-1 RMSE"),
    ("day1_mape", "Day-1 MAPE"),
]:
    s = ds[metric]
    win_rows.append({
        "metric": label,
        "global_better": s["improved_count"],
        "hybrid_better": s["worsened_count"],
        "tied": s["tied_count"],
        "mean_delta": s["mean_delta"],
        "std_delta": s["std_delta"],
    })
win_df = pd.DataFrame(win_rows)
display(Markdown("Table — Per-container outcomes (Δ = Global − Hybrid)"))
display(win_df.round(4))

mae_s = ds["day1_mae"]
print(
    f"MAE: Hybrid better on {mae_s['worsened_count']}/99 | "
    f"Global better on {mae_s['improved_count']}/99 | "
    f"ties {mae_s['tied_count']}"
)


Table — Per-container outcomes (Δ = Global − Hybrid)

,metric,global_better,hybrid_better,tied,mean_delta,std_delta
0,Day-1 MAE,32,67,0,0.1784,0.8416
1,Day-1 RMSE,22,77,0,0.2227,1.1360
2,Day-1 MAPE,33,66,0,4.5623,56.3618


MAE: Hybrid better on 67/99 | Global better on 32/99 | ties 0


## 5. Extreme and representative containers

In [5]:
sample_ids = select_comparison_sample_container_ids(deltas)
ranked = deltas.sort_values("delta_day1_mae").reset_index(drop=True)
best_g = ranked.iloc[0]
worst_g = ranked.iloc[-1]
median_row = ranked.iloc[len(ranked) // 2]
demo = deltas.loc[deltas["container_id"] == "c_11461"]

highlight = pd.DataFrame([
    {
        "role": "Best for Global (most negative ΔMAE)",
        "container_id": best_g["container_id"],
        "hybrid_mae": best_g["hybrid_day1_mae"],
        "global_mae": best_g["global_day1_mae"],
        "delta_mae": best_g["delta_day1_mae"],
    },
    {
        "role": "Worst for Global (most positive ΔMAE)",
        "container_id": worst_g["container_id"],
        "hybrid_mae": worst_g["hybrid_day1_mae"],
        "global_mae": worst_g["global_day1_mae"],
        "delta_mae": worst_g["delta_day1_mae"],
    },
    {
        "role": "Median ΔMAE container",
        "container_id": median_row["container_id"],
        "hybrid_mae": median_row["hybrid_day1_mae"],
        "global_mae": median_row["global_day1_mae"],
        "delta_mae": median_row["delta_day1_mae"],
    },
])
if not demo.empty:
    r = demo.iloc[0]
    highlight = pd.concat([
        highlight,
        pd.DataFrame([{
            "role": "Demo container",
            "container_id": r["container_id"],
            "hybrid_mae": r["hybrid_day1_mae"],
            "global_mae": r["global_day1_mae"],
            "delta_mae": r["delta_day1_mae"],
        }]),
    ], ignore_index=True)

display(Markdown("Table — Highlight containers"))
display(highlight.round(4))
print("Sample IDs used in frozen side-by-side panel:", sample_ids)


Table — Highlight containers

,role,container_id,hybrid_mae,global_mae,delta_mae
0,Best for Global (most negative ΔMAE),c_12237,17.6757,12.7636,-4.9121
1,Worst for Global (most positive ΔMAE),c_15640,4.0355,7.0663,3.0308
2,Median ΔMAE container,c_15604,0.7279,0.7563,0.0284
3,Demo container,c_11461,1.4544,2.2485,0.7941


Sample IDs used in frozen side-by-side panel: ['c_11461', 'c_12237', 'c_15640', 'c_15604']


## 6. Comparison plots (regenerated from frozen deltas)

Publication-style plots using the official comparison colors:
- Hybrid = `#C44E52`
- Global = `#4C72B0`


In [6]:
# --- ΔMAE histogram ---
delta_vals = deltas["delta_day1_mae"].dropna()
mean_delta = float(delta_vals.mean())

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(delta_vals, bins=20, color=COLOR_GLOBAL, edgecolor="black", alpha=0.85)
ax.axvline(0.0, color=COLOR_REFERENCE_LINE, linestyle="--", linewidth=1.5, label="No change (Δ = 0)")
ax.axvline(mean_delta, color=COLOR_HYBRID, linestyle="-", linewidth=1.5,
           label=f"Mean ΔMAE = {mean_delta:+.4f}%")
ax.set_title("Hybrid vs Global GRU — Per-Container Day-1 MAE Delta Distribution")
ax.set_xlabel("ΔMAE (Global − Hybrid, real CPU %)")
ax.set_ylabel("Container count")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

# --- MAE scatter ---
h_mae = deltas["hybrid_day1_mae"]
g_mae = deltas["global_day1_mae"]
fig, ax = plt.subplots(figsize=(7.5, 7))
ax.scatter(h_mae, g_mae, color=COLOR_GLOBAL, alpha=0.75, edgecolor="black", linewidth=0.4, s=42)
lo = float(min(h_mae.min(), g_mae.min()))
hi = float(max(h_mae.max(), g_mae.max()))
pad = 0.05 * (hi - lo) if hi > lo else 0.5
axis_min, axis_max = max(0.0, lo - pad), hi + pad
ax.plot([axis_min, axis_max], [axis_min, axis_max], color=COLOR_REFERENCE_LINE,
        linestyle="--", linewidth=1.5, label="Equal MAE (45°)")
ax.set_xlim(axis_min, axis_max)
ax.set_ylim(axis_min, axis_max)
ax.set_aspect("equal", adjustable="box")
ax.set_title("Hybrid vs Global GRU — Per-Container Day-1 MAE Scatter")
ax.set_xlabel("Hybrid Prophet + GRU Day-1 MAE (real CPU %)")
ax.set_ylabel("Global GRU Day-1 MAE (real CPU %)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

# --- MAE distribution overlay ---
h_vals = deltas["hybrid_day1_mae"].dropna()
g_vals = deltas["global_day1_mae"].dropna()
bins = np.linspace(float(min(h_vals.min(), g_vals.min())),
                   float(max(h_vals.max(), g_vals.max())), 21)
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(h_vals, bins=bins, color=COLOR_HYBRID, edgecolor="black", alpha=0.55,
        label=f"Hybrid (mean = {h_vals.mean():.4f}%)")
ax.hist(g_vals, bins=bins, color=COLOR_GLOBAL, edgecolor="black", alpha=0.55,
        label=f"Global GRU (mean = {g_vals.mean():.4f}%)")
ax.set_title("Hybrid vs Global GRU — Per-Container Day-1 MAE Distribution")
ax.set_xlabel("Day-1 MAE (real CPU %)")
ax.set_ylabel("Container count")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()


/var/folders/w6/h4_43dnn17d76hs8_g3x6vfm0000gn/T/ipykernel_30278/2379318159.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/w6/h4_43dnn17d76hs8_g3x6vfm0000gn/T/ipykernel_30278/2379318159.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/w6/h4_43dnn17d76hs8_g3x6vfm0000gn/T/ipykernel_30278/2379318159.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Frozen forecast visuals

Official side-by-side Actual vs Predicted figures from the comparison experiment (no re-inference).


In [7]:
frozen_plots = [
    ("Demo container c_11461", PLOT_DIR / "side_by_side" / "c_11461.png"),
    ("Best / worst / median sample panel", PLOT_DIR / "side_by_side_samples.png"),
]

for title, path in frozen_plots:
    if not path.exists():
        print(f"Missing plot: {path}")
        continue
    display(Markdown(f"### {title}"))
    img = mpimg.imread(path)
    fig, ax = plt.subplots(figsize=(12, 4.8))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(title)
    fig.tight_layout()
    plt.show()


### Demo container c_11461

/var/folders/w6/h4_43dnn17d76hs8_g3x6vfm0000gn/T/ipykernel_30278/902418115.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Best / worst / median sample panel

## 8. Per-container ΔMAE explorer

In [8]:
explorer = deltas[[
    "container_id",
    "hybrid_day1_mae", "global_day1_mae", "delta_day1_mae",
    "hybrid_day1_rmse", "global_day1_rmse", "delta_day1_rmse",
]].copy()
explorer["mae_winner"] = np.where(
    explorer["delta_day1_mae"] < 0, "Global",
    np.where(explorer["delta_day1_mae"] > 0, "Hybrid", "Tie"),
)
explorer = explorer.sort_values("delta_day1_mae")

display(Markdown("Table — Top 10 Global wins (most negative ΔMAE)"))
display(explorer.head(10).round(4))

display(Markdown("Table — Top 10 Hybrid wins (most positive ΔMAE)"))
display(explorer.tail(10).sort_values("delta_day1_mae", ascending=False).round(4))

# Optional focus container — change to inspect any container_id
FOCUS_CID = "c_11461"
focus = explorer.loc[explorer["container_id"] == FOCUS_CID]
display(Markdown(f"Table — Focus container `{FOCUS_CID}`"))
if focus.empty:
    display(pd.DataFrame({"note": [f"{FOCUS_CID} not in cohort"]}))
else:
    display(focus.round(4))


Table — Top 10 Global wins (most negative ΔMAE)

,container_id,hybrid_day1_mae,global_day1_mae,delta_day1_mae,hybrid_day1_rmse,global_day1_rmse,delta_day1_rmse,mae_winner
32,c_12237,17.6757,12.7636,-4.9121,22.8280,14.3926,-8.4354,Global
55,c_13615,5.1177,2.4918,-2.6259,5.6799,3.9670,-1.7129,Global
86,c_15527,6.0499,4.7181,-1.3318,7.2311,6.1005,-1.1306,Global
51,c_13389,7.3037,6.0262,-1.2775,8.2528,8.3573,0.1045,Global
41,c_12993,5.0878,4.3410,-0.7468,5.9064,5.2786,-0.6278,Global
15,c_11087,6.3413,6.1944,-0.1469,12.1436,11.8452,-0.2984,Global
69,c_14417,1.3130,1.1878,-0.1252,1.8680,1.5797,-0.2883,Global
94,c_15647,1.3395,1.2411,-0.0984,1.5954,1.6630,0.0676,Global
66,c_14396,2.3089,2.2276,-0.0813,2.7377,2.8151,0.0774,Global
52,c_13403,0.3621,0.3210,-0.0412,0.4677,0.4454,-0.0223,Global


Table — Top 10 Hybrid wins (most positive ΔMAE)

,container_id,hybrid_day1_mae,global_day1_mae,delta_day1_mae,hybrid_day1_rmse,global_day1_rmse,delta_day1_rmse,mae_winner
92,c_15640,4.0355,7.0663,3.0308,6.5570,8.6504,2.0934,Hybrid
31,c_12231,7.8592,10.0587,2.1995,8.6256,13.0842,4.4586,Hybrid
1,c_10034,3.6562,5.6031,1.9470,4.1073,6.5120,2.4047,Hybrid
72,c_14820,2.6026,4.5410,1.9384,3.9560,5.8940,1.9380,Hybrid
62,c_14135,2.0393,3.5478,1.5085,3.0834,4.2513,1.1680,Hybrid
74,c_14886,1.2293,2.6455,1.4162,1.5876,3.2865,1.6989,Hybrid
14,c_11013,2.7333,3.9077,1.1745,4.0416,4.9872,0.9455,Hybrid
81,c_15235,1.1802,2.3378,1.1575,1.5050,2.9782,1.4733,Hybrid
26,c_12158,1.9107,3.0602,1.1495,2.3090,4.0728,1.7638,Hybrid
71,c_14471,1.9474,2.9281,0.9807,2.5768,3.6312,1.0544,Hybrid


Table — Focus container `c_11461`

,container_id,hybrid_day1_mae,global_day1_mae,delta_day1_mae,hybrid_day1_rmse,global_day1_rmse,delta_day1_rmse,mae_winner
19,c_11461,1.4544,2.2485,0.7941,2.0343,3.0362,1.0018,Hybrid


## 9. Win-rate bar chart

In [9]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for ax, (_, row) in zip(axes, win_df.iterrows()):
    ax.bar(
        ["Hybrid better", "Global better"],
        [row["hybrid_better"], row["global_better"]],
        color=[COLOR_HYBRID, COLOR_GLOBAL],
        edgecolor="black",
    )
    ax.set_title(row["metric"])
    ax.set_ylabel("Containers" if ax is axes[0] else "")
    ax.set_ylim(0, 99)
    for i, v in enumerate([row["hybrid_better"], row["global_better"]]):
        ax.text(i, v + 1.5, str(int(v)), ha="center", fontsize=11)
fig.suptitle("Per-container win counts (99 containers)", y=1.02)
fig.tight_layout()
plt.show()


/var/folders/w6/h4_43dnn17d76hs8_g3x6vfm0000gn/T/ipykernel_30278/1075124320.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Deployment trade-offs

| Dimension | Hybrid Prophet + GRU | Global GRU |
|-----------|----------------------|------------|
| Accuracy (this study) | Lower mean MAE/RMSE | Higher mean MAE/RMSE |
| Win rate (MAE) | **67 / 99** | **32 / 99** |
| Inference complexity | Prophet fit per container + GRU | Single shared model predict |
| Operational surface | Global GRU + per-container Prophet state | One global GRU |
| New containers | Needs Prophet fit | Reuse frozen global model (in-distribution) |
| Failure modes | Occasional extreme Hybrid MAE outliers | More frequent modest regressions vs Hybrid |


## 11. Research answers (locked)

In [10]:
answers = f'''
### RQ1 — Overall Day-1 accuracy

**Finding:** Hybrid Prophet+GRU outperforms Global GRU on mean Day-1 MAE
({pr["day1_mae"]["hybrid_mean"]:.3f} vs {pr["day1_mae"]["global_mean"]:.3f})
and RMSE
({pr["day1_rmse"]["hybrid_mean"]:.3f} vs {pr["day1_rmse"]["global_mean"]:.3f}).

**Interpretation:** Explicit Prophet trend/seasonality reduces the burden on the residual GRU
for workloads with strong daily periodicity. Global GRU must learn level, variability, and
temporal structure jointly from the CPU sequence alone.

### RQ2 — Which containers favour which model?

**Finding:** Global is better on {mae_s["improved_count"]}/99 containers by MAE;
Hybrid is better on {mae_s["worsened_count"]}/99.

Largest Global win: `{best_g["container_id"]}` (ΔMAE = {best_g["delta_day1_mae"]:.3f}).  
Largest Hybrid win: `{worst_g["container_id"]}` (ΔMAE = {worst_g["delta_day1_mae"]:.3f}).

**Interpretation:** When Hybrid/Prophet decomposition fails, direct Global sequence modelling
can recover. When Hybrid already fits well, decomposition usually remains advantageous.

### RQ3 — Deployment trade-offs

Hybrid wins on **average accuracy**. Global wins on **deployment simplicity**
(one model, no per-container Prophet). The locked Module 1 choice for production forecasting
is Hybrid; Global remains the architecture baseline / ablation control.
'''
display(Markdown(answers))



### RQ1 — Overall Day-1 accuracy

**Finding:** Hybrid Prophet+GRU outperforms Global GRU on mean Day-1 MAE
(1.746 vs 1.924)
and RMSE
(2.388 vs 2.611).

**Interpretation:** Explicit Prophet trend/seasonality reduces the burden on the residual GRU
for workloads with strong daily periodicity. Global GRU must learn level, variability, and
temporal structure jointly from the CPU sequence alone.

### RQ2 — Which containers favour which model?

**Finding:** Global is better on 32/99 containers by MAE;
Hybrid is better on 67/99.

Largest Global win: `c_12237` (ΔMAE = -4.912).  
Largest Hybrid win: `c_15640` (ΔMAE = 3.031).

**Interpretation:** When Hybrid/Prophet decomposition fails, direct Global sequence modelling
can recover. When Hybrid already fits well, decomposition usually remains advantageous.

### RQ3 — Deployment trade-offs

Hybrid wins on **average accuracy**. Global wins on **deployment simplicity**
(one model, no per-container Prophet). The locked Module 1 choice for production forecasting
is Hybrid; Global remains the architecture baseline / ablation control.


## 12. Key findings

In [11]:
findings = f'''
1. Under the locked 99-container Day-1 protocol, **Hybrid is the better forecasting methodology**
   on mean MAE ({pr["day1_mae"]["hybrid_mean"]:.3f} vs {pr["day1_mae"]["global_mean"]:.3f},
   Δ = {pr["day1_mae"]["mean_delta_global_minus_hybrid"]:+.3f}).
2. Hybrid also wins on mean RMSE (Δ = {pr["day1_rmse"]["mean_delta_global_minus_hybrid"]:+.3f}).
3. Per-container MAE: Hybrid better **{mae_s["worsened_count"]}/99**, Global better **{mae_s["improved_count"]}/99**.
4. Global still wins on a meaningful minority — especially some high-error Hybrid outliers
   (e.g. `{best_g["container_id"]}`).
5. MAPE is secondary only (unstable near zero CPU); do not rank models by MAPE alone.
6. Conclusion is **configuration-specific** to these frozen Hybrid v1 and Global GRU v1 setups —
   not a claim that all hybrid models beat all GRUs.
'''
display(Markdown(findings))

print("Official summary path:")
print((EXP_DIR / "FINAL_COMPARISON_SUMMARY.md").relative_to(REPO_ROOT))



1. Under the locked 99-container Day-1 protocol, **Hybrid is the better forecasting methodology**
   on mean MAE (1.746 vs 1.924,
   Δ = +0.178).
2. Hybrid also wins on mean RMSE (Δ = +0.223).
3. Per-container MAE: Hybrid better **67/99**, Global better **32/99**.
4. Global still wins on a meaningful minority — especially some high-error Hybrid outliers
   (e.g. `c_12237`).
5. MAPE is secondary only (unstable near zero CPU); do not rank models by MAPE alone.
6. Conclusion is **configuration-specific** to these frozen Hybrid v1 and Global GRU v1 setups —
   not a claim that all hybrid models beat all GRUs.


Official summary path:
experiments/hybrid_vs_global_2026-07-17_095147/FINAL_COMPARISON_SUMMARY.md


## 13. Artifact index

| Artifact | Path |
|----------|------|
| Experiment | `experiments/hybrid_vs_global_2026-07-17_095147/` |
| Aggregate table | `comparison_table.csv` |
| Per-container deltas | `per_container_delta.csv` |
| Locked JSON | `phase6_comparison.json` |
| Executive summary | `FINAL_COMPARISON_SUMMARY.md` |
| Plots | `plots/` |
| Verification | `scripts/verify_hybrid_vs_global_comparison.py` (PASS) |
| Hybrid model notebook | `notebooks/hybrid_model.ipynb` |
| Global model notebook | `notebooks/global_gru_model.ipynb` |
